# Análise consolidada — Prompt Injection Review

Este notebook consolida os resultados de triagem por título/resumo provenientes de **IEEE** e **Springer**, valida as avaliações produzidas pelo `SCREENING_PROTOCOL_v1.0`, identifica pendências e duplicatas, calcula estatísticas descritivas e gera arquivos para a futura fase de leitura integral dos PDFs.

> Execute este notebook a partir da raiz do projeto ou da pasta `searches/`.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata

import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)

ROOT = next(
    (p for p in [Path.cwd(), Path.cwd() / "searches"]
     if (p / "IEEE").is_dir() and (p / "Springer").is_dir()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Execute a partir da raiz do projeto ou da pasta searches.")

IEEE_DIR = ROOT / "IEEE" / "IEEE-S1"
IEEE_RESULTS_DIR = IEEE_DIR / "IEEE_S1_batches_20" / "results"

SPRINGER_DIR = ROOT / "Springer"
SPRINGER_BATCH_DIR = SPRINGER_DIR / "SPRINGER_S1_batches_20"
SPRINGER_RESULTS_DIR = SPRINGER_BATCH_DIR / "results"

OUTPUT_DIR = ROOT / "analysis_outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("IEEE existe:", IEEE_DIR.exists())
print("Springer existe:", SPRINGER_DIR.exists())

## Configuração provisória para explorar a seleção

A rubrica não muda. O ponto de corte abaixo serve apenas para observar quantos estudos seriam encaminhados ao texto completo.

In [ ]:
FULL_TEXT_THRESHOLD = 0.85
LOW_CONFIDENCE_THRESHOLD = 0.60
GRAY_ZONE_MARGIN = 0.05

## Funções auxiliares

In [ ]:
def clean_text(value):
    if value is None or pd.isna(value):
        return ""
    return re.sub(r"\s+", " ", str(value)).strip()


def normalize_title(value):
    value = clean_text(value)
    value = value.replace(r"\{", "").replace(r"\}", "")
    value = value.replace("{", "").replace("}", "")
    value = value.replace("\\", "")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = value.lower()
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return re.sub(r"\s+", " ", value).strip()


def normalize_doi(value):
    value = clean_text(value).lower()
    value = re.sub(r"^https?://(?:dx\.)?doi\.org/", "", value)
    value = re.sub(r"^doi:\s*", "", value)
    return value.strip()


def split_bibtex_entries(text):
    entries = []
    pos = 0
    n = len(text)

    while pos < n:
        at = text.find("@", pos)
        if at == -1:
            break

        brace = text.find("{", at)
        if brace == -1:
            break

        depth = 0
        i = brace

        while i < n:
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    entries.append(text[at:i+1])
                    pos = i + 1
                    break
            i += 1
        else:
            raise ValueError(f"Entrada BibTeX não fechada a partir da posição {at}")

    return entries


def parse_bib_entry(raw):
    head = re.match(r"@([A-Za-z]+)\{([^,]+),", raw, flags=re.S)
    if not head:
        return {}

    entry_type = head.group(1).lower()
    citation_key = head.group(2).strip()
    pos = head.end()

    out = {"bib_key": citation_key, "entry_type": entry_type}

    while pos < len(raw):
        while pos < len(raw) and raw[pos] in " \t\r\n,":
            pos += 1

        if pos >= len(raw) or raw[pos] == "}":
            break

        match = re.match(r"([A-Za-z][A-Za-z0-9_-]*)\s*=\s*", raw[pos:])
        if not match:
            pos += 1
            continue

        field = match.group(1).lower()
        pos += match.end()

        if pos >= len(raw):
            break

        if raw[pos] == "{":
            depth = 1
            start = pos + 1
            pos += 1
            while pos < len(raw) and depth:
                if raw[pos] == "{":
                    depth += 1
                elif raw[pos] == "}":
                    depth -= 1
                pos += 1
            value = raw[start:pos-1]

        elif raw[pos] == '"':
            pos += 1
            start = pos
            while pos < len(raw):
                if raw[pos] == '"' and raw[pos-1] != "\\":
                    value = raw[start:pos]
                    pos += 1
                    break
                pos += 1
        else:
            start = pos
            while pos < len(raw) and raw[pos] not in ",}":
                pos += 1
            value = raw[start:pos]

        out[field] = clean_text(value)

    return out


def bib_files_to_dataframe(files, database):
    records = []

    for path in files:
        text = path.read_text(encoding="utf-8-sig", errors="replace")
        entries = split_bibtex_entries(text)

        for entry in entries:
            data = parse_bib_entry(entry)
            if not data:
                continue

            records.append({
                "database": database,
                "source_file": path.name,
                "bib_key": data.get("bib_key", ""),
                "entry_type": data.get("entry_type", ""),
                "title": data.get("title", ""),
                "abstract": data.get("abstract", ""),
                "authors": data.get("author", ""),
                "year": data.get("year", ""),
                "doi": normalize_doi(data.get("doi", "")),
                "url": data.get("url", ""),
                "venue": (
                    data.get("journal", "")
                    or data.get("booktitle", "")
                    or data.get("publisher", "")
                ),
                "content_type": data.get("type", ""),
            })

    df = pd.DataFrame(records)

    if not df.empty:
        df["title_norm"] = df["title"].map(normalize_title)

    return df


def read_jsonl_results(results_dir, database):
    rows = []

    if not results_dir.exists():
        print(
            f"[INFO] Pasta de resultados ainda não existe: "
            f"{results_dir}"
        )
        return pd.DataFrame()

    files = sorted(results_dir.glob("*.jsonl"))

    print(
        f"{database}: "
        f"{len(files)} arquivo(s) JSONL encontrado(s)"
    )

    for path in files:
        with path.open(
            "r",
            encoding="utf-8-sig"
        ) as f:

            for line_no, line in enumerate(f, start=1):

                line = line.strip()

                if not line:
                    continue

                try:
                    obj = json.loads(line)

                except json.JSONDecodeError as exc:

                    inicio = max(
                        0,
                        exc.pos - 150
                    )

                    fim = min(
                        len(line),
                        exc.pos + 150
                    )

                    trecho = line[inicio:fim]

                    raise ValueError(
                        "\n\nJSONL INVÁLIDO\n"
                        f"Base: {database}\n"
                        f"Arquivo: {path}\n"
                        f"Linha: {line_no}\n"
                        f"Coluna: {exc.colno}\n"
                        f"Erro: {exc.msg}\n\n"
                        f"Trecho:\n{trecho}\n\n"
                        "Corrija essa linha antes de continuar."
                    ) from exc

                scores = obj.get(
                    "scores",
                    {}
                ) or {}

                rows.append({
                    "database": database,

                    "evaluation_file":
                        path.name,

                    "doi": normalize_doi(obj.get("doi", "")),
                    "screening_id":
                        obj.get("id", ""),

                    "title":
                        obj.get("title", ""),

                    "title_norm":
                        normalize_title(
                            obj.get("title", "")
                        ),

                    "relevance_score":
                        obj.get(
                            "relevance_score"
                        ),

                    "confidence":
                        obj.get(
                            "confidence"
                        ),

                    "phenomenon_centrality":
                        scores.get(
                            "phenomenon_centrality"
                        ),

                    "rq_contribution":
                        scores.get(
                            "rq_contribution"
                        ),

                    "application_context":
                        scores.get(
                            "application_context"
                        ),

                    "evidence_or_synthesis":
                        scores.get(
                            "evidence_or_synthesis"
                        ),

                    "information_density":
                        scores.get(
                            "information_density"
                        ),

                    "hard_exclude":
                        obj.get(
                            "hard_exclude",
                            False
                        ),

                    "study_type":
                        obj.get(
                            "study_type",
                            ""
                        ),

                    "contribution_types":
                        obj.get(
                            "contribution_types",
                            []
                        ) or [],

                    "contexts":
                        obj.get(
                            "contexts",
                            []
                        ) or [],

                    "possible_duplicate_llm":
                        obj.get(
                            "possible_duplicate",
                            False
                        ),

                    "reason":
                        obj.get(
                            "reason",
                            ""
                        ),
                })

    return pd.DataFrame(rows)


def validate_screening_scores(df):
    if df.empty:
        return df.copy()

    out = df.copy()

    cols = [
        "phenomenon_centrality",
        "rq_contribution",
        "application_context",
        "evidence_or_synthesis",
        "information_density",
        "relevance_score",
        "confidence",
    ]

    for col in cols:
        out[col] = pd.to_numeric(out[col], errors="coerce")

    out["score_recalculated"] = (
        0.35 * out["phenomenon_centrality"]
        + 0.25 * out["rq_contribution"]
        + 0.15 * out["application_context"]
        + 0.15 * out["evidence_or_synthesis"]
        + 0.10 * out["information_density"]
    ).round(2)

    out.loc[out["hard_exclude"] == True, "score_recalculated"] = 0.00

    out["score_difference"] = (
        out["relevance_score"] - out["score_recalculated"]
    ).abs().round(4)

    out["score_formula_ok"] = out["score_difference"].fillna(999) <= 0.01

    return out


from difflib import SequenceMatcher


def smart_match_metadata_screening(
    metadata, screening, database="", fuzzy_threshold=0.90, min_margin=0.04
):
    """Match 1:1 por ID, DOI, titulo exato e fuzzy; ambiguidades ficam pendentes."""
    if not 0 <= fuzzy_threshold <= 1 or not 0 <= min_margin <= 1:
        raise ValueError("Os limites devem estar entre 0 e 1.")
    meta = metadata.copy().reset_index(drop=True)
    screen = screening.copy().reset_index(drop=True)
    for col in ["bib_key", "doi", "title"]:
        if col not in meta:
            meta[col] = ""
    for col in ["screening_id", "doi", "title", "relevance_score"]:
        if col not in screen:
            screen[col] = pd.Series(index=screen.index, dtype="object")
    meta["_meta_index"] = meta.index
    screen["_screen_index"] = screen.index

    def identifier(value):
        return clean_text(value).casefold()

    meta_keys = {
        "ID_EXACT": meta["bib_key"].map(identifier),
        "DOI_EXACT": meta["doi"].map(normalize_doi),
        "TITLE_EXACT": meta["title"].map(normalize_title),
    }
    screen_dois = screen["doi"].map(normalize_doi)
    id_dois = screen["screening_id"].map(normalize_doi)
    screen_dois = screen_dois.mask(
        screen_dois.eq("") & id_dois.str.match(r"^10\.\d{4,9}/\S+$"), id_dois
    )
    screen_keys = {
        "ID_EXACT": screen["screening_id"].map(identifier),
        "DOI_EXACT": screen_dois,
        "TITLE_EXACT": screen["title"].map(normalize_title),
    }
    links, used_meta, used_screen = [], set(), set()
    # Todas as correspondencias exatas precedem o fuzzy.
    # Um valor duplicado nao se torna unico por consumo de outra linha.
    for method in meta_keys:
        mk, sk = meta_keys[method], screen_keys[method]
        for si, key in sk.items():
            if si in used_screen or not key:
                continue
            candidates = mk.index[mk.eq(key)].tolist()
            if len(candidates) != 1 or sk.eq(key).sum() != 1:
                continue
            mi = candidates[0]
            if mi in used_meta:
                continue
            # DOI explicito conflitante exige revisao.
            md, sd = meta_keys["DOI_EXACT"][mi], screen_dois[si]
            if md and sd and md != sd:
                continue
            links.append(dict(_screen_index=si, _meta_index=mi,
                              match_method=method, match_score=1.0, match_margin=None))
            used_meta.add(mi)
            used_screen.add(si)

    def similarity(a, b):
        if not a or not b:
            return 0.0
        seq = SequenceMatcher(None, a, b).ratio()
        ta, tb = set(a.split()), set(b.split())
        overlap = len(ta & tb)
        combined = (0.60 * seq + 0.20 * overlap / len(ta | tb)
                    + 0.20 * overlap / min(len(ta), len(tb)))
        return max(seq, combined)

    suggestions, proposals = {}, {}
    for si, title in screen_keys["TITLE_EXACT"].items():
        if si in used_screen or not title:
            continue
        # Compara com todos os metadados para nao esconder candidatos ja usados.
        ranked = sorted(
            [(similarity(title, mt), mi)
             for mi, mt in meta_keys["TITLE_EXACT"].items() if mt],
            reverse=True,
        )
        if not ranked:
            continue
        score, mi = ranked[0]
        margin = score - ranked[1][0] if len(ranked) > 1 else score
        suggestions[si] = (mi, score, margin)
        md, sd = meta_keys["DOI_EXACT"][mi], screen_dois[si]
        if (score >= fuzzy_threshold and margin >= min_margin
                and mi not in used_meta and not (md and sd and md != sd)):
            proposals.setdefault(mi, []).append((si, score, margin))

    for mi, candidates in proposals.items():
        # Nao escolhe arbitrariamente entre avaliacoes concorrentes.
        if len(candidates) != 1:
            continue
        si, score, margin = candidates[0]
        links.append(dict(_screen_index=si, _meta_index=mi,
                          match_method="TITLE_FUZZY", match_score=score,
                          match_margin=margin))
        used_meta.add(mi)
        used_screen.add(si)

    link_columns = ["_screen_index", "_meta_index", "match_method",
                    "match_score", "match_margin"]
    links_df = pd.DataFrame(links, columns=link_columns)
    for col in ["_screen_index", "_meta_index"]:
        links_df[col] = links_df[col].astype("Int64")
    screen_for_merge = screen.drop(
        columns=["database", "title_norm"], errors="ignore"
    ).rename(columns={"title": "screening_title", "doi": "screening_doi"})
    result = meta.merge(links_df, on="_meta_index", how="left", validate="one_to_one")
    result = result.merge(
        screen_for_merge, on="_screen_index", how="left",
        suffixes=("", "_screening"), validate="many_to_one"
    )
    result["screening_status"] = result["relevance_score"].notna().map(
        {True: "EVALUATED", False: "PENDING"}
    )
    by_screen = {row["_screen_index"]: row for row in links}
    diagnostics = []
    for si, row in screen.iterrows():
        link = by_screen.get(si)
        mi, score, margin = suggestions.get(si, (None, 0.0, None))
        if link:
            mi, score, margin = (
                link["_meta_index"], link["match_score"], link["match_margin"]
            )
        diagnostics.append({
            "_screen_index": si, "database": database,
            "evaluation_file": row.get("evaluation_file", ""),
                    "screening_id": row["screening_id"], "screening_title": row["title"],
            "status": "MATCHED" if link else "UNMATCHED",
            "matched_title": meta.at[mi, "title"] if link else "",
            "best_candidate_title": meta.at[mi, "title"] if mi is not None else "",
            "best_candidate_doi": meta.at[mi, "doi"] if mi is not None else "",
            "match_method": link["match_method"] if link else "NONE",
            "match_score": score, "margin": margin,
        })
    diagnostic_columns = [
        "_screen_index", "database", "evaluation_file", "screening_id",
        "screening_title", "status", "matched_title", "best_candidate_title",
        "best_candidate_doi", "match_method", "match_score", "margin",
    ]
    return (
        result.drop(columns=["_meta_index", "_screen_index"]),
        pd.DataFrame(diagnostics, columns=diagnostic_columns),
    )


def merge_metadata_screening(metadata, screening):
    """Compatibilidade com chamadas anteriores."""
    return smart_match_metadata_screening(metadata, screening)[0]

def add_priority_band(df):
    out = df.copy()

    def band(score):
        if pd.isna(score):
            return "PENDING"
        if score >= 0.80:
            return "0.80–1.00 alta prioridade"
        if score >= 0.65:
            return "0.65–0.79 provável relevância"
        if score >= 0.50:
            return "0.50–0.64 zona cinzenta"
        if score >= 0.25:
            return "0.25–0.49 baixa prioridade"
        return "0.00–0.24 provável exclusão"

    out["priority_band"] = out["relevance_score"].map(band)
    return out


def make_dedup_key(row):
    doi = normalize_doi(row.get("doi", ""))
    if doi:
        return "doi:" + doi
    return "title:" + normalize_title(row.get("title", ""))


def explode_counts(df, column):
    if df.empty or column not in df:
        return pd.Series(dtype=int)

    values = df[column].explode().dropna().astype(str)
    values = values[values.str.len() > 0]
    return values.value_counts()


def first_nonempty(series):
    for value in series:
        if isinstance(value, list):
            if value:
                return value
        elif pd.notna(value) and str(value).strip():
            return value
    return ""

## Carregar os BibTeX

In [ ]:
# IEEE: apenas os cinco arquivos originais da exportação.
ieee_master_pattern = re.compile(r"^IEEE_S1_\d{3}-\d{3}\.bib$", re.I)

ieee_bib_files = sorted(
    p for p in IEEE_DIR.glob("*.bib")
    if ieee_master_pattern.match(p.name)
)

# Springer: o BibTeX foi produzido diretamente em batches.
springer_bib_files = sorted(
    SPRINGER_BATCH_DIR.glob("SPRINGER_S1_batch_*.bib")
)

ieee_meta = bib_files_to_dataframe(ieee_bib_files, "IEEE")
springer_meta = bib_files_to_dataframe(springer_bib_files, "Springer")

print("IEEE arquivos:", len(ieee_bib_files), "| registros:", len(ieee_meta))
print("Springer arquivos:", len(springer_bib_files), "| registros:", len(springer_meta))

## Ler e validar os JSONL já produzidos

In [ ]:
from pathlib import Path
import json

def validar_jsonl_pasta(pasta):
    erros = []
    total_linhas = 0
    total_validas = 0

    arquivos = sorted(Path(pasta).glob("*.jsonl"))

    print(f"Arquivos encontrados: {len(arquivos)}\n")

    for arquivo in arquivos:
        validas_arquivo = 0
        erros_arquivo = 0

        with arquivo.open("r", encoding="utf-8-sig") as f:
            for numero_linha, linha in enumerate(f, start=1):
                linha = linha.strip()

                if not linha:
                    continue

                total_linhas += 1

                try:
                    json.loads(linha)
                    total_validas += 1
                    validas_arquivo += 1

                except json.JSONDecodeError as exc:
                    erros_arquivo += 1

                    inicio = max(0, exc.pos - 120)
                    fim = min(len(linha), exc.pos + 120)

                    trecho = linha[inicio:fim]

                    erros.append({
                        "arquivo": arquivo,
                        "linha": numero_linha,
                        "erro": exc.msg,
                        "coluna": exc.colno,
                        "posicao": exc.pos,
                        "trecho": trecho,
                        "conteudo_completo": linha,
                    })

                    print("=" * 80)
                    print(f"ERRO: {arquivo.name}")
                    print(f"Linha: {numero_linha}")
                    print(f"Coluna: {exc.colno}")
                    print(f"Mensagem: {exc.msg}")
                    print()
                    print("Trecho ao redor do erro:")
                    print(trecho)
                    print()
                    print(" " * min(120, exc.pos - inicio) + "^")
                    print("=" * 80)
                    print()

        print(
            f"{arquivo.name}: "
            f"{validas_arquivo} válidas, "
            f"{erros_arquivo} inválidas"
        )

    print("\nRESUMO")
    print("-" * 40)
    print("Linhas com conteúdo:", total_linhas)
    print("JSON válidos:", total_validas)
    print("JSON inválidos:", len(erros))

    return erros


erros_ieee = validar_jsonl_pasta(IEEE_RESULTS_DIR)

In [ ]:
ieee_screen = validate_screening_scores(
    read_jsonl_results(IEEE_RESULTS_DIR, "IEEE")
)

springer_screen = validate_screening_scores(
    read_jsonl_results(SPRINGER_RESULTS_DIR, "Springer")
)

print("IEEE avaliações encontradas:", len(ieee_screen))
print("Springer avaliações encontradas:", len(springer_screen))

validation = pd.concat(
    [
        x for x in [ieee_screen, springer_screen]
        if not x.empty
    ],
    ignore_index=True,
) if (not ieee_screen.empty or not springer_screen.empty) else pd.DataFrame()

invalid_scores = (
    validation.loc[~validation["score_formula_ok"]].copy()
    if not validation.empty
    else pd.DataFrame()
)

print("Avaliações com divergência da fórmula:", len(invalid_scores))

if len(invalid_scores):
    display(
        invalid_scores[
            [
                "database",
                "evaluation_file",
                "title",
                "relevance_score",
                "score_recalculated",
                "score_difference",
            ]
        ]
    )

## Consolidar metadados + avaliações

In [ ]:
ieee, ieee_match_diagnostics = smart_match_metadata_screening(ieee_meta, ieee_screen, "IEEE")
springer, springer_match_diagnostics = smart_match_metadata_screening(springer_meta, springer_screen, "Springer")

combined = pd.concat([ieee, springer], ignore_index=True, sort=False)
combined = add_priority_band(combined)
combined["dedup_key"] = combined.apply(make_dedup_key, axis=1)

summary_status = (
    combined.groupby(["database", "screening_status"])
    .size()
    .unstack(fill_value=0)
)

summary_status["TOTAL"] = summary_status.sum(axis=1)
display(summary_status)

print("Total:", len(combined))
print("Avaliados:", (combined["screening_status"] == "EVALUATED").sum())
print("Pendentes:", (combined["screening_status"] == "PENDING").sum())

## Verificar JSONL que não casaram com os BibTeX

In [ ]:
match_diagnostics = pd.concat(
    [ieee_match_diagnostics, springer_match_diagnostics], ignore_index=True
)
unmatched_ieee = ieee_match_diagnostics.loc[
    ieee_match_diagnostics["status"].eq("UNMATCHED")
].copy()
unmatched_springer = springer_match_diagnostics.loc[
    springer_match_diagnostics["status"].eq("UNMATCHED")
].copy()

print("IEEE sem correspondencia:", len(unmatched_ieee))
print("Springer sem correspondencia:", len(unmatched_springer))
display(match_diagnostics.groupby(["database", "match_method"]).size().to_frame("n"))
display(match_diagnostics.loc[match_diagnostics["status"].eq("UNMATCHED")])

## Detectar duplicatas por DOI ou título

In [ ]:
dup_mask = combined.duplicated("dedup_key", keep=False)

duplicates = (
    combined.loc[dup_mask]
    .sort_values(["dedup_key", "database", "title"])
    .copy()
)

print("Registros em grupos duplicados:", len(duplicates))
print("Grupos duplicados:", duplicates["dedup_key"].nunique())

if len(duplicates):
    display(
        duplicates[
            [
                "database",
                "title",
                "doi",
                "year",
                "screening_status",
                "relevance_score",
                "dedup_key",
            ]
        ]
    )

## Criar dataset em nível de estudo único

In [ ]:
study_rows = []

for key, group in combined.groupby("dedup_key", dropna=False):
    evaluated = group.loc[group["screening_status"] == "EVALUATED"].copy()
    scores = pd.to_numeric(evaluated["relevance_score"], errors="coerce").dropna()

    score = scores.median() if len(scores) else None
    score_range = scores.max() - scores.min() if len(scores) > 1 else 0.0

    study_rows.append({
        "study_key": key,
        "sources": sorted(group["database"].dropna().unique().tolist()),
        "source_count": group["database"].nunique(),
        "title": first_nonempty(group["title"]),
        "abstract": first_nonempty(group["abstract"]),
        "authors": first_nonempty(group["authors"]),
        "year": first_nonempty(group["year"]),
        "doi": first_nonempty(group["doi"]),
        "url": first_nonempty(group["url"]),
        "venue": first_nonempty(group["venue"]),
        "screening_status": "EVALUATED" if len(evaluated) else "PENDING",
        "n_evaluations": len(evaluated),
        "relevance_score": score,
        "score_range_if_repeated": round(float(score_range), 4) if len(scores) else None,
        "confidence": (
            pd.to_numeric(evaluated["confidence"], errors="coerce").median()
            if len(evaluated) else None
        ),
        "hard_exclude": (
            bool(evaluated["hard_exclude"].fillna(False).all())
            if len(evaluated) else False
        ),
        "study_type": first_nonempty(evaluated["study_type"]) if len(evaluated) else "",
        "contribution_types": sorted({
            item
            for values in evaluated["contribution_types"]
            if isinstance(values, list)
            for item in values
        }),
        "contexts": sorted({
            item
            for values in evaluated["contexts"]
            if isinstance(values, list)
            for item in values
        }),
        "reason": first_nonempty(evaluated["reason"]) if len(evaluated) else "",
    })

studies = pd.DataFrame(study_rows)
studies = add_priority_band(studies)

print("Registros brutos:", len(combined))
print("Estudos únicos:", len(studies))
print("Duplicatas consolidadas:", len(combined) - len(studies))

## Insights do screening

In [ ]:
evaluated_studies = studies.loc[
    studies["screening_status"] == "EVALUATED"
].copy()

print("Estudos únicos avaliados:", len(evaluated_studies))

if len(evaluated_studies):
    display(
        evaluated_studies["relevance_score"]
        .describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95])
        .to_frame("relevance_score")
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(
        evaluated_studies["relevance_score"].dropna(),
        bins=[i / 20 for i in range(21)],
        edgecolor="black",
    )
    ax.set_title("Distribuição dos relevance_score")
    ax.set_xlabel("relevance_score")
    ax.set_ylabel("Número de estudos")
    plt.show()

    print("Tipos de estudo:")
    display(
        evaluated_studies["study_type"]
        .replace("", "UNCLASSIFIED")
        .value_counts()
        .to_frame("n")
    )

    print("Contextos:")
    display(explode_counts(combined.loc[combined["screening_status"]=="EVALUATED"], "contexts").to_frame("n"))

    print("Contribuições:")
    display(explode_counts(combined.loc[combined["screening_status"]=="EVALUATED"], "contribution_types").to_frame("n"))

## Simular candidatos para leitura integral

In [ ]:
full_text_candidates = (
    evaluated_studies.loc[
        (evaluated_studies["hard_exclude"] == False)
        & (evaluated_studies["relevance_score"] >= FULL_TEXT_THRESHOLD)
    ]
    .sort_values(["relevance_score", "confidence"], ascending=[False, False])
    .copy()
)

gray_zone = (
    evaluated_studies.loc[
        (evaluated_studies["hard_exclude"] == False)
        & (
            evaluated_studies["relevance_score"].between(
                FULL_TEXT_THRESHOLD - GRAY_ZONE_MARGIN,
                FULL_TEXT_THRESHOLD + GRAY_ZONE_MARGIN,
                inclusive="both",
            )
            | (evaluated_studies["confidence"].fillna(0) < LOW_CONFIDENCE_THRESHOLD)
        )
    ]
    .sort_values(["relevance_score", "confidence"], ascending=[False, True])
    .copy()
)

print(
    f"Candidatos provisórios (score >= {FULL_TEXT_THRESHOLD:.2f}):",
    len(full_text_candidates),
)
print("Zona de revisão manual:", len(gray_zone))

display(
    full_text_candidates[
        [
            "title",
            "sources",
            "year",
            "doi",
            "relevance_score",
            "confidence",
            "study_type",
            "contexts",
            "contribution_types",
            "reason",
        ]
    ].head(50)
)

## Ranking atual

In [ ]:
top_ranked = (
    evaluated_studies.loc[evaluated_studies["hard_exclude"] == False]
    .sort_values(["relevance_score", "confidence"], ascending=[False, False])
)

display(
    top_ranked[
        [
            "title",
            "sources",
            "year",
            "doi",
            "relevance_score",
            "confidence",
            "study_type",
            "contexts",
            "contribution_types",
            "reason",
        ]
    ].head(30)
)

## Pendências

In [ ]:
pending = (
    studies.loc[studies["screening_status"] == "PENDING"]
    .sort_values(["sources", "title"])
    .copy()
)

print("Estudos únicos pendentes:", len(pending))
display(pending[["title", "sources", "year", "doi", "url"]].head(50))

## Resumo por base

In [ ]:
rows = []

for database, group in combined.groupby("database"):
    eg = group.loc[group["screening_status"] == "EVALUATED"]
    scores = pd.to_numeric(eg["relevance_score"], errors="coerce")

    rows.append({
        "database": database,
        "records_total": len(group),
        "evaluated": len(eg),
        "pending": (group["screening_status"] == "PENDING").sum(),
        "hard_exclude": eg["hard_exclude"].fillna(False).sum() if len(eg) else 0,
        "mean_score": scores.mean() if len(eg) else None,
        "median_score": scores.median() if len(eg) else None,
        "score_ge_threshold": (scores >= FULL_TEXT_THRESHOLD).sum() if len(eg) else 0,
    })

database_summary = pd.DataFrame(rows)
display(database_summary)

## Exportar arquivos

In [ ]:
threshold_tag = str(FULL_TEXT_THRESHOLD).replace(".", "_")

combined.to_csv(
    OUTPUT_DIR / "all_records_with_screening.csv",
    index=False,
    encoding="utf-8-sig",
)

studies.to_csv(
    OUTPUT_DIR / "master_studies_deduplicated.csv",
    index=False,
    encoding="utf-8-sig",
)

top_ranked.to_csv(
    OUTPUT_DIR / "ranked_evaluated_studies.csv",
    index=False,
    encoding="utf-8-sig",
)

full_text_candidates.to_csv(
    OUTPUT_DIR / f"full_text_candidates_{threshold_tag}.csv",
    index=False,
    encoding="utf-8-sig",
)

gray_zone.to_csv(
    OUTPUT_DIR / f"manual_review_zone_{threshold_tag}.csv",
    index=False,
    encoding="utf-8-sig",
)

pending.to_csv(
    OUTPUT_DIR / "pending_screening.csv",
    index=False,
    encoding="utf-8-sig",
)

duplicates.to_csv(
    OUTPUT_DIR / "possible_duplicates.csv",
    index=False,
    encoding="utf-8-sig",
)

invalid_scores.to_csv(
    OUTPUT_DIR / "screening_validation_issues.csv",
    index=False,
    encoding="utf-8-sig",
)

database_summary.to_csv(
    OUTPUT_DIR / "summary_by_database.csv",
    index=False,
    encoding="utf-8-sig",
)

print("Arquivos gerados em:", OUTPUT_DIR.resolve())
for p in sorted(OUTPUT_DIR.glob("*.csv")):
    print(" -", p.name)
match_diagnostics.to_csv(
    OUTPUT_DIR / "matching_diagnostics.csv", index=False, encoding="utf-8-sig"
)

## Uso metodológico

Enquanto o screening estiver incompleto, use o notebook apenas para:

- acompanhar quantos estudos já foram avaliados;
- observar a distribuição dos scores;
- verificar erros de cálculo;
- identificar duplicatas;
- observar tipos de estudos, contextos e contribuições;
- experimentar diferentes pontos de corte.

O corte definitivo para leitura integral deve ser definido somente depois que as bases planejadas tiverem sido processadas.

O arquivo central para a fase seguinte será:

`analysis_outputs/master_studies_deduplicated.csv`

In [ ]:
# ============================================================
# GERAR CSV FINAL CONSOLIDADO
# ============================================================

final_df = studies.copy()

# ------------------------------------------------------------
# 1. Criar status de decisão provisória
# ------------------------------------------------------------

def classify_screening(row):
    if row["screening_status"] == "PENDING":
        return "PENDING"

    if row["hard_exclude"] == True:
        return "HARD_EXCLUDE"

    score = row["relevance_score"]

    if pd.isna(score):
        return "PENDING"

    if score >= FULL_TEXT_THRESHOLD:
        return "FULL_TEXT_CANDIDATE"

    if score >= (FULL_TEXT_THRESHOLD - GRAY_ZONE_MARGIN):
        return "MANUAL_REVIEW"

    return "BELOW_THRESHOLD"


final_df["selection_status"] = final_df.apply(
    classify_screening,
    axis=1
)


# ------------------------------------------------------------
# 2. Criar ranking
# ------------------------------------------------------------

# Ranking somente entre estudos avaliados e não hard-excluded
eligible_mask = (
    (final_df["screening_status"] == "EVALUATED")
    & (final_df["hard_exclude"] == False)
)

final_df["relevance_rank"] = pd.NA

final_df.loc[
    eligible_mask,
    "relevance_rank"
] = (
    final_df.loc[
        eligible_mask,
        "relevance_score"
    ]
    .rank(
        method="min",
        ascending=False
    )
    .astype("Int64")
)


# ------------------------------------------------------------
# 3. Converter campos que são listas para texto legível no CSV
# ------------------------------------------------------------

def list_to_text(value):
    if isinstance(value, list):
        return "; ".join(map(str, value))
    return value


for column in [
    "sources",
    "contexts",
    "contribution_types",
]:
    if column in final_df.columns:
        final_df[column] = final_df[column].apply(
            list_to_text
        )


# ------------------------------------------------------------
# 4. Organizar as colunas
# ------------------------------------------------------------

preferred_columns = [
    "relevance_rank",
    "selection_status",
    "relevance_score",
    "confidence",

    "title",
    "authors",
    "year",
    "doi",
    "url",
    "venue",

    "sources",
    "source_count",

    "study_type",
    "contribution_types",
    "contexts",

    "hard_exclude",
    "screening_status",

    "n_evaluations",
    "score_range_if_repeated",

    "reason",
    "abstract",

    "study_key",
]

# Mantém somente colunas existentes
preferred_columns = [
    col
    for col in preferred_columns
    if col in final_df.columns
]

final_df = final_df[
    preferred_columns
].copy()


# ------------------------------------------------------------
# 5. Definir ordem dos status
# ------------------------------------------------------------

status_order = {
    "FULL_TEXT_CANDIDATE": 1,
    "MANUAL_REVIEW": 2,
    "BELOW_THRESHOLD": 3,
    "HARD_EXCLUDE": 4,
    "PENDING": 5,
}

final_df["_status_order"] = (
    final_df["selection_status"]
    .map(status_order)
    .fillna(99)
)


# ------------------------------------------------------------
# 6. Ordenar
# ------------------------------------------------------------

final_df = (
    final_df
    .sort_values(
        by=[
            "_status_order",
            "relevance_score",
            "confidence",
        ],
        ascending=[
            True,
            False,
            False,
        ],
        na_position="last",
    )
    .drop(
        columns="_status_order"
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# 7. Exportar
# ------------------------------------------------------------

output_file = (
    OUTPUT_DIR
    / "FINAL_screening_studies.csv"
)

final_df.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig",
)


# ------------------------------------------------------------
# 8. Mostrar resumo
# ------------------------------------------------------------

print("=" * 70)
print("CSV FINAL GERADO")
print("=" * 70)

print(f"Arquivo: {output_file.resolve()}")
print(f"Total de estudos únicos: {len(final_df)}")

print("\nSituação:")

display(
    final_df[
        "selection_status"
    ]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="n")
)

print(
    "\nCandidatos para leitura integral:",
    (
        final_df["selection_status"]
        == "FULL_TEXT_CANDIDATE"
    ).sum()
)

print(
    "Revisão manual:",
    (
        final_df["selection_status"]
        == "MANUAL_REVIEW"
    ).sum()
)

print(
    "Abaixo do corte:",
    (
        final_df["selection_status"]
        == "BELOW_THRESHOLD"
    ).sum()
)

print(
    "Hard exclude:",
    (
        final_df["selection_status"]
        == "HARD_EXCLUDE"
    ).sum()
)

print(
    "Ainda pendentes:",
    (
        final_df["selection_status"]
        == "PENDING"
    ).sum()
)

print("\nPrimeiros registros:")

display(
    final_df[
        [
            "relevance_rank",
            "selection_status",
            "relevance_score",
            "confidence",
            "title",
            "sources",
            "study_type",
        ]
    ].head(20)
)